<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day12-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 12, Segment 2 discussion — Row-wise vs. column-wise attention, actually coded

The book page *describes* AlphaFold's Evoformer applying attention along two different
axes of an MSA-shaped array (sequences x positions): row-wise (across positions, within
one sequence) and column-wise (across sequences, at one fixed position) — but it never
shows the mechanism in code. This notebook builds a tiny, real version of exactly that.

In [1]:
import math
import numpy as np

np.random.seed(0)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(X):
    """Plain self-attention over the second-to-last axis of X (..., n, d)."""
    scores = X @ np.swapaxes(X, -1, -2) / math.sqrt(X.shape[-1])
    weights = softmax(scores, axis=-1)
    return weights @ X, weights

# A toy "MSA": 4 sequences, 6 aligned positions, an 8-dim per-residue representation
n_seqs, n_pos, d = 4, 6, 8
msa_repr = np.random.randn(n_seqs, n_pos, d)
print("MSA representation shape (sequences, positions, features):", msa_repr.shape)

MSA representation shape (sequences, positions, features): (4, 6, 8)


## Row-wise attention: across positions, within each sequence separately

This is exactly the same self-attention as Segment 1's sentence example — the "sequence" axis here just stands in for the sentence's token axis.

In [2]:
row_out, row_weights = self_attention(msa_repr)  # attends over the position axis, per sequence
print("row-wise output shape:", row_out.shape)
print("row-wise weight matrix shape (one 6x6 attention pattern per sequence):", row_weights.shape)
print("each row of each sequence's attention pattern sums to 1.0:", np.allclose(row_weights.sum(axis=-1), 1.0))

row-wise output shape: (4, 6, 8)
row-wise weight matrix shape (one 6x6 attention pattern per sequence): (4, 6, 6)
each row of each sequence's attention pattern sums to 1.0: True


## Column-wise attention: across sequences, at each fixed position

Transpose the sequence and position axes first, run the *same* self-attention function, then transpose back — this is the whole trick. Column-wise attention is what lets information about a coevolving pair of positions flow across the different sequences in the MSA, at a fixed position — conceptually the same coevolution signal Day 5's profile methods and Day 10's contact prediction already used, just processed by attention instead of counting statistics or a CNN.

In [3]:
msa_repr_T = msa_repr.transpose(1, 0, 2)  # (positions, sequences, d) -- swap the two axes
col_out_T, col_weights = self_attention(msa_repr_T)  # now attends over the SEQUENCE axis, per position
col_out = col_out_T.transpose(1, 0, 2)  # back to (sequences, positions, d)

print("column-wise output shape:", col_out.shape)
print("column-wise weight matrix shape (one 4x4 attention pattern per position):", col_weights.shape)
print("row-wise and column-wise outputs are different:", not np.allclose(row_out, col_out))

column-wise output shape: (4, 6, 8)
column-wise weight matrix shape (one 4x4 attention pattern per position): (6, 4, 4)
row-wise and column-wise outputs are different: True


## What this actually shows

Both `self_attention` calls are the exact same function — the only thing that changes
between "row-wise" and "column-wise" is *which axis gets transposed to the front* before
calling it. That's the real mechanical content behind the book page's Evoformer paragraph:
row-wise and column-wise attention are not two different algorithms, they're the same
attention mechanism applied to two different reshapings of the same array. Verified
directly above: the two really do produce different outputs (`row_out` != `col_out`) from
the exact same input, and both weight matrices are genuine probability distributions
(each row sums to 1.0).